# 10 -- Model comparison and champion selection

Load the tuned champion from every family (notebooks 06-09), evaluate them
on the **untouched holdout**, compare discrimination and calibration, and
pick the best model.

Tuned configurations live in `src.modeling.champions` (the GridSearchCV
winners). Fitted models are cached in `models/` via `src.modeling.registry`,
so the first run trains and saves each model and later runs just load them.

Families compared:
- Linear: logistic regression
- Tree: decision tree
- Ensemble: random forest
- Boosting: XGBoost, LightGBM, CatBoost

In [ ]:
from __future__ import annotations

import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODEL_COMPARISON_PATH
from src.modeling import (
    build_all_champions,
    classification_metrics,
    evaluate_split,
    has_model,
    load_model,
    load_modeling_data,
    make_pipeline,
    metrics_frame,
    save_model,
)
from src.modeling import plots
from src.modeling.champions import FAMILY, SEARCH_CV_AUC

data = load_modeling_data()
pos_weight = data.pos_weight
pd.Series(data.summary())

## Load or train each family champion

Each champion is a full pipeline (`preprocess` + estimator) trained on the
entire training partition. Cached models are loaded from `models/`; missing
ones are trained once and saved. First run can take ~10-20 min on a laptop.

In [ ]:
champs = build_all_champions(pos_weight)
fitted = {}
for name, est in champs.items():
    if has_model(name):
        print(f"load  {name}")
        fitted[name] = load_model(name)
        continue
    print(f"train {name} ...", flush=True)
    t = time.perf_counter()
    pipe = make_pipeline(est)
    pipe.fit(data.X_train, data.y_train)
    metrics = evaluate_split(pipe, data.X_holdout, data.y_holdout)
    save_model(
        name, pipe, metrics=metrics,
        extra={"family": FAMILY.get(name), "search_cv_auc": SEARCH_CV_AUC.get(name)},
    )
    fitted[name] = pipe
    print(f"  done in {time.perf_counter() - t:.1f}s  holdout_auc={metrics['roc_auc']:.4f}")
print("ready:", list(fitted))

## Holdout comparison table

Discrimination (AUC, Gini, KS, PR-AUC) and calibration (log loss, Brier) on
the untouched holdout, with each family's search-sample CV AUC for context.

In [ ]:
rows = {name: evaluate_split(m, data.X_holdout, data.y_holdout) for name, m in fitted.items()}
comparison = metrics_frame(rows)
comparison.insert(0, "family", [FAMILY.get(n) for n in comparison.index])
comparison["search_cv_auc"] = [SEARCH_CV_AUC.get(n) for n in comparison.index]
comparison = comparison.sort_values("roc_auc", ascending=False)
comparison.round(4)

## ROC and precision-recall overlay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in comparison.index:
    proba = fitted[name].predict_proba(data.X_holdout)[:, 1]
    plots.plot_roc_pr(data.y_holdout.to_numpy(), proba, label=name, axes=axes)
plt.tight_layout()
plt.show()

## Calibration overlay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for name in comparison.index:
    proba = fitted[name].predict_proba(data.X_holdout)[:, 1]
    plots.plot_calibration(data.y_holdout.to_numpy(), proba, ax=ax, label=name)
plt.show()

## Metric bar charts (AUC and KS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comparison["roc_auc"].plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Holdout ROC AUC")
axes[0].invert_yaxis()
comparison["ks"].plot.barh(ax=axes[1], color="teal")
axes[1].set_title("Holdout KS")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## Select and persist the champion

The best model by holdout AUC is tagged as `champion` in the registry and a
comparison summary is written to `reports/model_comparison.json`. Note: the
official `reports/metrics.json` is produced by the runnable pipeline in
Stage 6, not by this exploratory notebook.

In [ ]:
best_name = comparison.index[0]
best_metrics = evaluate_split(fitted[best_name], data.X_holdout, data.y_holdout)
save_model(
    "champion", fitted[best_name], metrics=best_metrics,
    extra={"family": FAMILY.get(best_name), "source_model": best_name},
)

summary = {
    "champion": best_name,
    "champion_family": FAMILY.get(best_name),
    "holdout_metrics": {k: round(float(v), 6) for k, v in best_metrics.items()},
    "all_models": {
        name: {k: round(float(v), 6) for k, v in rows[name].items()}
        for name in comparison.index
    },
    "n_holdout": int(len(data.y_holdout)),
}
MODEL_COMPARISON_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_COMPARISON_PATH.write_text(json.dumps(summary, indent=2), encoding="ascii")
print("champion:", best_name, "(", FAMILY.get(best_name), ")")
print("holdout AUC:", round(best_metrics["roc_auc"], 4), "KS:", round(best_metrics["ks"], 4))
print("wrote", MODEL_COMPARISON_PATH)

## Takeaways

- The comparison ranks all family champions on the same untouched holdout,
  so the choice is apples-to-apples.
- Boosting models (XGBoost / LightGBM / CatBoost) are expected to lead on
  AUC/KS, with logistic regression as the interpretable baseline.
- The selected champion is cached as `models/champion.joblib` and will be
  ported into `src/train.py` for Stage 6, where `reports/metrics.json` is
  generated by the runnable pipeline.